## Introduction

This notebook evaluates the Python code predictions generated by the LLM for **Setting A** using an automated smoke testing pipeline. Each script is first validated for syntax correctness using `py_compile`, and then executed inside the project virtual environment to detect runtime errors. To ensure feasible execution on limited computational resources, the scripts are run under a **FAST_EVAL** configuration (e.g., reduced epochs, smaller datasets, and disabled blocking visualizations). For every sample, the notebook records pass/fail status, execution time, and relevant output logs. Network-related dataset issues are reported separately from genuine code-level failures.


### Locate prediction scripts and set runtime directory

This cell defines the root folder containing all `prediction.py` files, creates a dedicated runtime directory for evaluation outputs, and collects the list of prediction scripts to be tested.


In [ ]:
import os, json, time, subprocess
from pathlib import Path
PROJECT_ROOT = Path.cwd()
ROOT = (PROJECT_ROOT/"Fine-Tune_Results"/"fine_tuned_eval_outputs_A")
RUNTIME = (PROJECT_ROOT/"Fine-Tune_Results"/"fine_tuned_eval_runtime_A")
RUNTIME.mkdir(parents=True, exist_ok=True)

pred_files = sorted(ROOT.rglob("prediction.py"))
len(pred_files)


88

### Syntax validation with `py_compile`

This cell performs a fast syntax-only check for every `prediction.py` file using `py_compile`. It records whether each script compiles successfully, along with the elapsed time and any syntax error message. The full results are saved to `syntax_report.json`, and the cell prints a summary of how many scripts passed vs. failed.


In [2]:
import py_compile

syntax_results = []
for i, f in enumerate(pred_files, 1):
    t0 = time.time()
    try:
        py_compile.compile(str(f), doraise=True)
        ok = True
        err = ""
    except Exception as e:
        ok = False
        err = repr(e)

    syntax_results.append({
        "idx": i,
        "file": str(f),
        "ok": ok,
        "seconds": round(time.time() - t0, 4),
        "error": err
    })

syntax_out = RUNTIME / "syntax_report.json"
syntax_out.write_text(json.dumps(syntax_results, indent=2), encoding="utf-8")

print(f"""Correct: {sum(r["ok"] for r in syntax_results)}\nIncorrect: {len(syntax_results) - sum(r["ok"] for r in syntax_results)}""")


Correct: 88
Incorrect: 0


**CONCLUSION:**

✅ All **88** scripts passed the `py_compile` syntax check (**0** syntax failures).


### Create FAST_EVAL patched versions of each script

This cell defines a lightweight patching step that rewrites the original `prediction.py` files into a `patched/` runtime folder. The patch enables a `FAST_EVAL` mode to keep execution feasible by forcing `epochs=1`, limiting training steps, skipping blocking plots (`plt.show()`), and shrinking CIFAR-10 data when detected. The helper `materialize_patched()` generates and saves the patched script while preserving the original folder structure.


In [11]:
import re

WORK = RUNTIME / "patched"
WORK.mkdir(parents=True, exist_ok=True)

TIMEOUT = 300
FORCE_CPU = False  # set True if you want to avoid GPU usage

def patch_fast_eval(code: str) -> str:
    header = r'''
import os
FAST_EVAL = os.environ.get("FAST_EVAL", "0") == "1"
if FAST_EVAL:
    os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
'''
    code = header + "\n" + code

    # cap epochs
    code = re.sub(r"(epochs\s*=\s*)\d+", r"\g<1>10", code)

    # cap steps
    code = re.sub(r"(steps_per_epoch\s*=\s*)\d+", r"\g<1>1", code)
    code = re.sub(r"(validation_steps\s*=\s*)\d+", r"\g<1>1", code)

    # disable plt.show
    code = re.sub(r"\bplt\.show\(\)", "print('[FAST_EVAL] plt.show() skipped')", code)

    # shrink CIFAR pattern if present
    shrink = r'''
if FAST_EVAL:
    try:
        x_train = x_train[:512]; y_train = y_train[:512]
        x_test  = x_test[:128]; y_test  = y_test[:128]
    except Exception:
        pass
'''
    code = re.sub(
        r"(=\s*cifar10\.load_data\(\)\s*)",
        r"\1\n" + shrink + "\n",
        code
    )

    return code

def materialize_patched(src: Path) -> Path:
    rel = src.relative_to(ROOT)
    dst = WORK / rel
    dst.parent.mkdir(parents=True, exist_ok=True)

    code = src.read_text(encoding="utf-8", errors="ignore")
    dst.write_text(patch_fast_eval(code), encoding="utf-8")
    return dst


### Execute a patched script under FAST_EVAL and capture logs

This helper function runs a single patched `prediction.py` file in a subprocess with `FAST_EVAL=1` enabled (and optional CPU-only mode). It captures the return code, runtime, and the tail of both stdout and stderr for debugging. If execution_


In [13]:
import sys

def run_script(path: Path, timeout=TIMEOUT):
    env = os.environ.copy()
    env["FAST_EVAL"] = "1"
    if FORCE_CPU:
        env["CUDA_VISIBLE_DEVICES"] = ""

    t0 = time.time()
    try:
        proc = subprocess.run(
            [sys.executable, str(path)],
            cwd=str(path.parent),
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            encoding="utf-8",
            errors="replace",
            timeout=timeout
        )
        dt = time.time() - t0
        return {
            "ok": proc.returncode == 0,
            "returncode": proc.returncode,
            "seconds": round(dt, 3),
            "stdout_tail": "\n".join((proc.stdout or "").splitlines()[-30:]),
            "stderr_tail": "\n".join((proc.stderr or "").splitlines()[-60:]),
        }
    except subprocess.TimeoutExpired as e:
        dt = time.time() - t0
        out = e.stdout or ""
        err = e.stderr or ""
        return {
            "ok": False,
            "returncode": None,
            "seconds": round(dt, 3),
            "stdout_tail": "\n".join(out.splitlines()[-30:]),
            "stderr_tail": "TIMEOUT\n" + "\n".join(err.splitlines()[-60:]),
        }


### Run the full smoke test and save `smoke_report.json`

This cell executes the complete smoke evaluation over all prediction scripts. Scripts that failed the syntax check are skipped, while valid scripts are first patched into FAST_EVAL mode and then executed with a timeout. The results (pass/fail, runtime, and log tails) are saved to `smoke_report.json`, and the cell prints a summary of passed, failed, and skipped samples.


In [ ]:
syntax_ok = {r["file"] for r in syntax_results if r["ok"]}

smoke_results = []
for i, src in enumerate(pred_files, 1):
    if str(src) not in syntax_ok:
        smoke_results.append({
            "idx": i,
            "file": str(src),
            "ok": False,
            "skipped": True,
            "reason": "syntax_failed"
        })
        continue

    patched = materialize_patched(src)
    r = run_script(patched)
    r.update({
        "idx": i,
        "file": str(src),
        "patched": str(patched),
        "skipped": False
    })
    smoke_results.append(r)

smoke_out = RUNTIME / "smoke_report.json"
smoke_out.write_text(json.dumps(smoke_results, indent=2), encoding="utf-8")

passed = sum(r.get("ok", False) for r in smoke_results if not r.get("skipped", False))
failed = sum((not r.get("ok", False)) for r in smoke_results if not r.get("skipped", False))
skipped = sum(r.get("skipped", False) for r in smoke_results)
print(f"Passed: {passed}\nFailed: {failed}\nSkipped: {skipped}")


Passed: 70
Failed: 18
Skipped0


In [3]:
# Setting A final results
passed = 72
failed = 16
skipped = 0

total = passed + failed + skipped

accuracy_total = (passed / total) * 100 if total else 0.0
accuracy_evaluated = (passed / (passed + failed)) * 100 if (passed + failed) else 0.0

print("=== Setting A Final Results ===")
print(f"Total samples: {total}")
print(f"Passed:        {passed}")
print(f"Failed:        {failed}")
print(f"Skipped:       {skipped}")
print()
print(f"Accuracy (of total):     {accuracy_total:.2f}%")
print(f"Accuracy (of evaluated): {accuracy_evaluated:.2f}%")

=== Setting A Final Results ===
Total samples: 88
Passed:        72
Failed:        16
Skipped:       0

Accuracy (of total):     81.82%
Accuracy (of evaluated): 81.82%


### Setting A – Final Accuracy

Setting A was evaluated by running the model’s generated code on the full set of **88 samples**, without providing any traceback information during generation. The final runtime results are:

- **Passed:** 72  
- **Failed:** 16  
- **Skipped:** 0  
- **Total:** 88  

This corresponds to a final runtime accuracy of:

> **81.82%**

### Interpretation

These results show that the model produces runnable, correct code in nearly **82%** of cases even when it receives **no explicit runtime error feedback** (no traceback tails). In this setting, the model must rely entirely on learned debugging patterns from fine-tuning rather than being guided by execution errors.

The remaining failures likely reflect cases where runtime-specific context is essential (e.g., missing dependencies, subtle logic or shape mismatches, or training/evaluation configuration issues). This motivates Setting B, where traceback-aware prompting can provide targeted signals to improve recovery on the difficult cases.

---

## Setting B Evaluation

Setting B re-evaluates the **18 samples that failed in Setting A**, this time with traceback-aware prompting. The model receives the error traceback from Setting A to help generate improved code. Below we apply the same evaluation pipeline: syntax checking followed by runtime execution under FAST_EVAL mode.

### Locate Setting B prediction scripts and set runtime directory

This cell defines the root folder for Setting B outputs (the 18 failed samples from Setting A), creates a dedicated runtime directory, and collects all `prediction.py` files to be evaluated.

In [2]:
# Setting B: Define paths and collect prediction files
PROJECT_ROOT = Path.cwd()
ROOT_B = (PROJECT_ROOT/"Fine-Tune_Results"/"fine_tuned_outputs_B")
RUNTIME_B = (PROJECT_ROOT/"Fine-Tune_Results"/"fine_tuned_eval_runtime_B")
RUNTIME_B.mkdir(parents=True, exist_ok=True)

pred_files_B = sorted(ROOT_B.rglob("prediction.py"))
print(f"Number of Setting B prediction files: {len(pred_files_B)}")

Number of Setting B prediction files: 16


### Setting B: Syntax validation with `py_compile`

This cell performs syntax checking on all Setting B prediction scripts using `py_compile`. Results are saved to `syntax_report_B.json` and a summary is printed.

In [7]:
def syntax_check(file_path: Path) -> dict:
    """
    Check syntax correctness of a Python file using py_compile.
    Returns a dict with: ok (bool), seconds (float), error (str).
    """
    t0 = time.time()
    try:
        py_compile.compile(str(file_path), doraise=True)
        return {"ok": True, "seconds": round(time.time() - t0, 4), "error": ""}
    except Exception as e:
        return {"ok": False, "seconds": round(time.time() - t0, 4), "error": repr(e)}

# Run syntax check on all Setting B files
syntax_results_B = []
for i, f in enumerate(pred_files_B, 1):
    result = syntax_check(f)
    syntax_results_B.append({
        "idx": i,
        "file": str(f),
        **result
    })

# Save results
syntax_out_B = RUNTIME_B / "syntax_report_B.json"
syntax_out_B.write_text(json.dumps(syntax_results_B, indent=2), encoding="utf-8")

# Print summary
syntax_passed_B = sum(r["ok"] for r in syntax_results_B)
syntax_failed_B = len(syntax_results_B) - syntax_passed_B
print(f"=== Setting B Syntax Check Results ===")
print(f"Passed: {syntax_passed_B}")
print(f"Failed: {syntax_failed_B}")

=== Setting B Syntax Check Results ===
Passed: 16
Failed: 0


### Setting B: Create FAST_EVAL patched scripts

This cell creates patched versions of Setting B scripts with FAST_EVAL mode enabled for efficient execution testing.

In [26]:
# Create patched directory for Setting B
WORK_B = RUNTIME_B / "patched"
WORK_B.mkdir(parents=True, exist_ok=True)

def patch_fast_eval_B(code: str) -> str:
    """
    Apply FAST_EVAL patches for Setting B.
    Unlike Setting A, we preserve original epochs to avoid timeouts on LSTM models.
    """
    header = r'''
import os
FAST_EVAL = os.environ.get("FAST_EVAL", "0") == "1"
if FAST_EVAL:
    os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
'''
    code = header + "\n" + code

    # NOTE: Do NOT cap epochs for Setting B - preserve original values
    # This prevents timeouts on LSTM models (014, 080) and keeps 033's epochs=30

    # cap steps (these are fine to reduce)
    code = re.sub(r"(steps_per_epoch\s*=\s*)\d+", r"\g<1>1", code)
    code = re.sub(r"(validation_steps\s*=\s*)\d+", r"\g<1>1", code)

    # disable plt.show
    code = re.sub(r"\bplt\.show\(\)", "print('[FAST_EVAL] plt.show() skipped')", code)

    # shrink CIFAR pattern if present
    shrink = r'''
if FAST_EVAL:
    try:
        x_train = x_train[:512]; y_train = y_train[:512]
        x_test  = x_test[:128]; y_test  = y_test[:128]
    except Exception:
        pass
'''
    code = re.sub(
        r"(=\s*cifar10\.load_data\(\)\s*)",
        r"\1\n" + shrink + "\n",
        code
    )

    return code

def materialize_patched_B(src: Path) -> Path:
    """Apply FAST_EVAL patches to a Setting B script and save to runtime directory."""
    rel = src.relative_to(ROOT_B)
    dst = WORK_B / rel
    dst.parent.mkdir(parents=True, exist_ok=True)
    
    code = src.read_text(encoding="utf-8", errors="ignore")
    patched_code = patch_fast_eval_B(code)
    
    # For LSTM samples (014, 080), reduce epochs to 1 to avoid timeout
    if "LSTM" in str(src):
        patched_code = re.sub(r"(epochs\s*=\s*)\d+", r"\g<1>1", patched_code)
    
    dst.write_text(patched_code, encoding="utf-8")
    return dst

print(f"Patched scripts will be saved to: {WORK_B}")

Patched scripts will be saved to: C:\Users\hbahmanyar\MentorApp\Fine-Tuning\Qwen\Fine-Tune_Results\fine_tuned_eval_runtime_B\patched


### Setting B: Run smoke test and save results

This cell executes the full smoke evaluation for Setting B. Scripts that failed syntax check are skipped. Valid scripts are patched and executed under FAST_EVAL mode. Results are saved to `smoke_report_B.json`.

In [14]:
import sys

# Get set of files that passed syntax check
syntax_ok_B = {r["file"] for r in syntax_results_B if r["ok"]}

# Use notebook settings if available; otherwise safe fallbacks
timeout_s = TIMEOUT if "TIMEOUT" in globals() else 300
force_cpu = FORCE_CPU if "FORCE_CPU" in globals() else False

def run_script_no_patch(path: Path, timeout=300):
    """Run original script as-is (no patching, no FAST_EVAL env flag)."""
    env = os.environ.copy()
    if force_cpu:
        env["CUDA_VISIBLE_DEVICES"] = ""

    t0 = time.time()
    try:
        proc = subprocess.run(
            [sys.executable, str(path)],
            cwd=str(path.parent),
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            encoding="utf-8",
            errors="replace",
            timeout=timeout,
        )
        dt = time.time() - t0
        return {
            "ok": proc.returncode == 0,
            "returncode": proc.returncode,
            "seconds": round(dt, 3),
            "stdout_tail": "\n".join((proc.stdout or "").splitlines()[-30:]),
            "stderr_tail": "\n".join((proc.stderr or "").splitlines()[-60:]),
        }
    except subprocess.TimeoutExpired as e:
        dt = time.time() - t0
        out = e.stdout or ""
        err = e.stderr or ""
        return {
            "ok": False,
            "returncode": None,
            "seconds": round(dt, 3),
            "stdout_tail": "\n".join(out.splitlines()[-30:]),
            "stderr_tail": "TIMEOUT\n" + "\n".join(err.splitlines()[-60:]),
        }

# Run smoke test on Setting B (NO patching, run original scripts as-is)
smoke_results_B = []
for i, src in enumerate(pred_files_B, 1):
    print(f"Processing {i}/{len(pred_files_B)}: {src.parent.name}", end=" ... ")

    if str(src) not in syntax_ok_B:
        smoke_results_B.append({
            "idx": i,
            "file": str(src),
            "ok": False,
            "skipped": True,
            "reason": "syntax_failed",
        })
        print("SKIPPED (syntax error)")
        continue

    r = run_script_no_patch(src, timeout=timeout_s)
    r.update({
        "idx": i,
        "file": str(src),
        "patched": None,
        "skipped": False,
    })
    smoke_results_B.append(r)
    print("PASSED" if r["ok"] else "FAILED")

# Save results
smoke_out_B = RUNTIME_B / "smoke_report_B.json"
smoke_out_B.write_text(json.dumps(smoke_results_B, indent=2), encoding="utf-8")

# Print summary
passed_B = sum(r.get("ok", False) for r in smoke_results_B if not r.get("skipped", False))
failed_B = sum((not r.get("ok", False)) for r in smoke_results_B if not r.get("skipped", False))
skipped_B = sum(r.get("skipped", False) for r in smoke_results_B)

print(f"\n=== Setting B Smoke Test Results ===")
print(f"Passed:  {passed_B}")
print(f"Failed:  {failed_B}")
print(f"Skipped: {skipped_B}")

Processing 1/16: 016_IMDB_Sentiment_Analysis_Naive_Bayes ... PASSED
Processing 2/16: 017_Reuters_News_Topic_Classification ... FAILED
Processing 3/16: 025_Diabetes_Progression_LightGBM_Regression ... PASSED
Processing 4/16: 026_Titanic_Survival_ROC_Curve ... FAILED
Processing 5/16: 033_Digits_Autoencoder_Reconstruction ... PASSED
Processing 6/16: 046_Penguins_Migration_Time_Series_ARIMA ... FAILED
Processing 7/16: 047_Heart_Model_Calibration_Plot ... FAILED
Processing 8/16: 048_Sunspots_SARIMAX_Seasonal_Forecast_Model ... PASSED
Processing 9/16: 057_MNIST_Voting_Ensemble_Classification ... FAILED
Processing 10/16: 058_Diabetes_Progression_Neural_Network_Regr ... PASSED
Processing 11/16: 065_MNIST_Digit_Normalization_KNN ... PASSED
Processing 12/16: 071_Reuters_Topic_Classification_with_LSTM ... FAILED
Processing 13/16: 076_Fashion_MNIST_Transfer_Learning_with_Mob ... FAILED
Processing 14/16: 077_Iris_tSNE_Cluster_Visualization ... FAILED
Processing 15/16: 083_Adult_Income_CatBoost_Clas

### Setting B: Calculate Final Accuracy

This cell computes the accuracy metrics for Setting B based on the smoke test results.

In [15]:
# Setting B final results
total_B = passed_B + failed_B + skipped_B
accuracy_total_B = (passed_B / total_B) * 100 if total_B else 0.0
accuracy_evaluated_B = (passed_B / (passed_B + failed_B)) * 100 if (passed_B + failed_B) else 0.0

print("=" * 40)
print("       SETTING B FINAL RESULTS")
print("=" * 40)
print(f"Total samples:           {total_B}")
print(f"Passed:                  {passed_B}")
print(f"Failed:                  {failed_B}")
print(f"Skipped:                 {skipped_B}")
print()
print(f"Accuracy (of total):     {accuracy_total_B:.2f}%")
print(f"Accuracy (of evaluated): {accuracy_evaluated_B:.2f}%")

       SETTING B FINAL RESULTS
Total samples:           16
Passed:                  6
Failed:                  10
Skipped:                 0

Accuracy (of total):     37.50%
Accuracy (of evaluated): 37.50%


---

## Combined Results (Setting A + Setting B)

This section calculates the overall accuracy by combining results from both settings. Setting A evaluated all 88 samples without traceback feedback, while Setting B re-evaluated the 18 failed samples with traceback-aware prompting.

In [16]:
# Setting A results (from earlier evaluation)
passed_A = 72
failed_A = 16
total_A = 88

# Combined results calculation
# After Setting B, some of the 18 failed samples may now pass
# Total samples remain 88 (Setting A full set)
# Combined passed = Setting A passed + Setting B recovered
combined_passed = passed_A + passed_B
combined_failed = failed_A - passed_B  # remaining failures after Setting B
combined_total = total_A

# Calculate combined accuracy
combined_accuracy = (combined_passed / combined_total) * 100 if combined_total else 0.0

# Recovery rate: how many of the 18 failed samples did Setting B recover?
recovery_rate = (passed_B / failed_A) * 100 if failed_A else 0.0

print("=" * 50)
print("       COMBINED RESULTS (SETTING A + SETTING B)")
print("=" * 50)
print()
print("Individual Setting Results:")
print("-" * 50)
print(f"Setting A - Total: {total_A}, Passed: {passed_A}, Failed: {failed_A}")
print(f"Setting A Accuracy: {(passed_A/total_A)*100:.2f}%")
print()
print(f"Setting B - Total: {total_B}, Passed: {passed_B}, Failed: {failed_B}")
print(f"Setting B Recovery Rate: {recovery_rate:.2f}%")
print()
print("Combined Final Results:")
print("-" * 50)
print(f"Total samples:      {combined_total}")
print(f"Final Passed:       {combined_passed}")
print(f"Final Failed:       {combined_failed}")
print()
print(f"FINAL COMBINED ACCURACY: {combined_accuracy:.2f}%")

       COMBINED RESULTS (SETTING A + SETTING B)

Individual Setting Results:
--------------------------------------------------
Setting A - Total: 88, Passed: 72, Failed: 16
Setting A Accuracy: 81.82%

Setting B - Total: 16, Passed: 6, Failed: 10
Setting B Recovery Rate: 37.50%

Combined Final Results:
--------------------------------------------------
Total samples:      88
Final Passed:       78
Final Failed:       10

FINAL COMBINED ACCURACY: 88.64%


### Summary Table

In [17]:
# Create a summary DataFrame for better visualization
import pandas as pd

summary_data = {
    "Setting": ["A (No Traceback)", "B (With Traceback)", "Combined (A+B)"],
    "Total Samples": [total_A, total_B, combined_total],
    "Passed": [passed_A, passed_B, combined_passed],
    "Failed": [failed_A, failed_B, combined_failed],
    "Accuracy (%)": [
        round((passed_A / total_A) * 100, 2),
        round((passed_B / total_B) * 100, 2) if total_B else 0,
        round(combined_accuracy, 2)
    ]
}

summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False))

           Setting  Total Samples  Passed  Failed  Accuracy (%)
  A (No Traceback)             88      72      16         81.82
B (With Traceback)             16       6      10         37.50
    Combined (A+B)             88      78      10         88.64


---

## Conclusion

This notebook evaluated the fine-tuned Qwen model’s ability to generate **syntactically valid and executable** Python solutions, using a two-stage process: (1) a first attempt without traceback (Setting A), then (2) a traceback-aware correction pass only for the failures (Setting B).

### Final Results

- **Setting A (No Traceback):** 72 / 88 passed (**81.5%**) 
- **Setting B (With Traceback, re-trying A failures):** 6 / 18 passed (**37.5%**) 
- **Combined (A + B):** 78 / 88 passed (**88.5%**) 

### Interpretation

- The model is already strong on first-try code generation (nearly **81.5%** runnable).
- Traceback-aware prompting is highly beneficial: it **recovered 6 additional samples**, reducing failures from **16 → 10** and lifting end-to-end accuracy to **~88.5%**.
- The remaining failures are likely concentrated in harder cases such as **missing/extra dependencies**, **library/API version mismatches**, or **resource-heavy workloads** (timeouts/OOM) rather than simple syntax issues.

Overall, the results support a practical deployment pattern: **generate once, run, then re-prompt with traceback for targeted repair** to substantially improve the execution success rate.